# One-step прогноз (BTC-USD, ETH-USD) на 1m / 1h / 1d
Модели: Naive, DLinear, NHITS, iTransformer (NeuralForecast / PyTorch)

Цель: one-step прогноз цены. Модели обучаются на log-returns, затем прогноз переводится обратно в цену.
Данные: Binance Vision (market-data-only), Close.


In [2]:
#@title Установка зависимостей (Colab)
!pip -q install -U pip
!pip -q install yfinance==0.2.40 neuralforecast requests
print("Готово. Runtime → Restart runtime, затем запускай сверху вниз.")


Готово. Runtime → Restart runtime, затем запускай сверху вниз.


In [3]:
#@title Импорты и базовые настройки
import os, time, warnings, math
from dataclasses import dataclass

import numpy as np
import pandas as pd

import requests
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
np.random.seed(42)

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, DLinear, iTransformer


In [4]:
#@title Настройки эксперимента (под ~2 часа)
TICKERS = ["BTC-USD", "ETH-USD"]
STEPS   = ["1m", "1h", "1d"]
SEED    = 42
TEST_RATIO = 0.2

LOOKBACK_BY_STEP = {
    "1m": 240,   # 4 часа
    "1h": 168,   # 7 суток
    "1d": 365,   # 1 год
}

DAYS_BY_STEP = {
    "1m": 7,     # минутки
    "1h": 730,   # 2 года
    "1d": 3650,  # 10 лет
}

BINANCE_SYMBOL = {
    "BTC-USD": "BTCUSDT",
    "ETH-USD": "ETHUSDT",
}

FREQ_BY_STEP = {"1m": "min", "1h": "h", "1d": "D"}

MAX_STEPS_BY_MODEL = {
    "DLinear":      {"1m": 800,  "1h": 800,  "1d": 800},
    "NHITS":        {"1m": 1200, "1h": 1100, "1d": 1000},
    "iTransformer": {"1m": 600,  "1h": 700,  "1d": 800},
}

BATCH_BY_STEP = {"1m": 512, "1h": 256, "1d": 128}
LR = 1e-3


In [9]:
#@title Вспомогательные функции (метрики + загрузка Binance Vision)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

BINANCE_BASE = "https://data-api.binance.vision"
KLINES_PATH  = "/api/v3/klines"

INTERVAL_MS = {"1m": 60_000, "1h": 3_600_000, "1d": 86_400_000}

def _binance_get(url, params, timeout=30):
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r.json()

def download_close_binance(symbol: str, interval: str, days: int, sleep_s: float = 0.12) -> pd.DataFrame:
    """Возвращает DataFrame ['unique_id','ds','y'] для NeuralForecast."""
    assert interval in INTERVAL_MS, f"Unsupported interval {interval}"
    end_ms = int(time.time() * 1000)
    start_ms = end_ms - int(days * 24 * 3600 * 1000)

    url = BINANCE_BASE + KLINES_PATH

    all_rows = []
    cur = start_ms
    limit = 1000
    step_ms = INTERVAL_MS[interval] * limit

    while cur < end_ms:
        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": cur,
            "endTime": min(cur + step_ms - 1, end_ms),
            "limit": limit,
        }
        data = _binance_get(url, params)
        if not data:
            break

        all_rows.extend(data)
        last_open = int(data[-1][0])
        cur = last_open + INTERVAL_MS[interval]

        time.sleep(sleep_s)

        # защита от зависания
        if len(data) == 1 and last_open >= end_ms - INTERVAL_MS[interval]:
            break

    if len(all_rows) == 0:
        raise RuntimeError(f"Пустые данные (Binance) symbol={symbol} interval={interval} days={days}")

    df = pd.DataFrame(all_rows, columns=[
        "open_time","open","high","low","close","volume","close_time",
        "qav","num_trades","taker_base","taker_quote","ignore"
    ])

    df["ds"] = pd.to_datetime(df["open_time"].astype(np.int64), unit="ms", utc=True)
    df["y"]  = df["close"].astype(float)

    out = df[["ds","y"]].dropna().sort_values("ds").reset_index(drop=True)
    out.insert(0, "unique_id", symbol)
    return out

def train_test_split_df(df: pd.DataFrame, test_ratio: float):
    n = len(df)
    n_test = max(2, int(round(n * test_ratio)))  # минимум 2, чтобы были лаги
    n_train = n - n_test
    return df.iloc[:n_train].copy(), df.iloc[n_train:].copy()

def to_log_returns(df: pd.DataFrame) -> pd.DataFrame:
    """df: unique_id, ds, y(price) -> y becomes log-return. Сохраняем цену и prev цену."""
    out = df.copy()
    out["y_prev"] = out.groupby("unique_id")["y"].shift(1)
    out["r"] = np.log(out["y"] / out["y_prev"])
    out = out.dropna(subset=["r"]).copy()
    out["y_price"] = out["y"]
    out["y"] = out["r"].astype(float)
    return out[["unique_id","ds","y","y_price","y_prev"]].reset_index(drop=True)

def returns_to_price(y_prev: np.ndarray, r_hat: np.ndarray) -> np.ndarray:
    return y_prev * np.exp(r_hat)

def naive_one_step_price(y_train_price: np.ndarray, y_test_price: np.ndarray):
    y_hat = np.empty_like(y_test_price)
    prev = y_train_price[-1]
    for i in range(len(y_test_price)):
        y_hat[i] = prev
        prev = y_test_price[i]
    return y_hat


In [10]:
#@title Запуск одной модели (rolling one-step backtest по цене)

@dataclass
class RunResult:
    ticker: str
    step: str
    model: str
    input_size: int
    days: int
    n_train: int
    n_test: int
    rmse_test: float
    mae_test: float
    mape_test: float
    train_seconds: float

def run_nf_model_one_step_price(model_name: str, step: str, df_train_lr: pd.DataFrame, df_test_lr: pd.DataFrame, input_size: int):
    h = 1
    max_steps = int(MAX_STEPS_BY_MODEL[model_name][step])
    batch_size = int(BATCH_BY_STEP[step])

    common = dict(
        h=h,
        input_size=input_size,
        max_steps=max_steps,
        learning_rate=LR,
        batch_size=batch_size,
        random_seed=SEED,
    )

    if model_name == "DLinear":
        model = DLinear(**common)
    elif model_name == "NHITS":
        model = NHITS(**common)
    elif model_name == "iTransformer":
        model = iTransformer(n_series=1, **common)  # univariate
    else:
        raise ValueError(model_name)

    nf = NeuralForecast(models=[model], freq=FREQ_BY_STEP[step])

    df_all = pd.concat([df_train_lr, df_test_lr], ignore_index=True)

    t0 = time.time()
    cv = nf.cross_validation(df=df_all[["unique_id","ds","y"]], test_size=len(df_test_lr), n_windows=None, step_size=1)
    t_train = time.time() - t0

    r_hat = cv[model_name].values

    # восстановление цены
    y_prev = df_test_lr["y_prev"].values[-len(r_hat):]
    y_true_price = df_test_lr["y_price"].values[-len(r_hat):]
    y_hat_price = returns_to_price(y_prev, r_hat)

    return y_true_price, y_hat_price, t_train

def run_one_experiment(ticker: str, step: str, model: str):
    symbol = BINANCE_SYMBOL[ticker]
    days = int(DAYS_BY_STEP[step])
    input_size = int(LOOKBACK_BY_STEP[step])

    df_raw = download_close_binance(symbol, interval=step, days=days)
    df_raw = df_raw.copy()
    df_raw["unique_id"] = ticker

    df_lr = to_log_returns(df_raw)
    df_train_lr, df_test_lr = train_test_split_df(df_lr, TEST_RATIO)

    if model == "Naive":
        y_train_price = df_train_lr["y_price"].values
        y_test_price  = df_test_lr["y_price"].values
        t0 = time.time()
        y_hat_price = naive_one_step_price(y_train_price, y_test_price)
        t_train = 0.0
        y_true_price = y_test_price
    else:
        y_true_price, y_hat_price, t_train = run_nf_model_one_step_price(model, step, df_train_lr, df_test_lr, input_size=input_size)

    return RunResult(
        ticker=ticker,
        step=step,
        model=model,
        input_size=input_size,
        days=days,
        n_train=len(df_train_lr),
        n_test=len(df_test_lr),
        rmse_test=rmse(y_true_price, y_hat_price),
        mae_test=mae(y_true_price, y_hat_price),
        mape_test=mape(y_true_price, y_hat_price),
        train_seconds=float(t_train),
    )


In [11]:
#@title Запуск всех прогонов и сохранение registry.csv
MODELS = ["Naive", "DLinear", "NHITS", "iTransformer"]

rows = []
total = len(TICKERS) * len(STEPS) * len(MODELS)
done = 0

for ticker in TICKERS:
    for step in STEPS:
        for model in MODELS:
            done += 1
            print(f"[{done}/{total}] {ticker} step={step} model={model}")
            try:
                r = run_one_experiment(ticker, step, model)
                rows.append(r.__dict__)
                print(f"   TEST(price): RMSE={r.rmse_test:.6f} MAE={r.mae_test:.6f} MAPE={r.mape_test:.4f}% | train={r.train_seconds:.1f}s | n={r.n_test}")
            except Exception as e:
                print(f"   [SKIP] -> {e}")

registry = pd.DataFrame(rows)
display(registry.sort_values(["ticker","step","rmse_test"]))

registry.to_csv("registry.csv", index=False)
print(f"Сохранено: registry.csv (строк: {len(registry)})")


[1/24] BTC-USD step=1m model=Naive
   TEST(price): RMSE=58.905498 MAE=41.079013 MAPE=0.0583% | train=0.0s | n=2016
[2/24] BTC-USD step=1m model=DLinear


INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss          │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train  │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler        │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ decomp        │ SeriesDecomp  │      0 │ train │     0 │
│ 4 │ linear_trend  │ Linear        │    241 │ train │     0 │
│ 5 │ linear_season │ Linear        │    241 │ train │     0 │
└───┴───────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 482                                                                                              
Non-trainable params: 0                                                                                            
Total params: 482                                                                                                  
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 8                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

   TEST(price): RMSE=128.797238 MAE=91.929373 MAPE=3.0461% | train=31.3s | n=620


,ticker,step,model,input_size,days,n_train,n_test,rmse_test,mae_test,mape_test,train_seconds
8,BTC-USD,1d,Naive,365,3650,2478,620,2084.853757,1487.286823,1.706705,0.000000
9,BTC-USD,1d,DLinear,365,3650,2478,620,2220.849684,1606.910540,1.840387,24.274297
11,BTC-USD,1d,iTransformer,365,3650,2478,620,2263.839677,1612.385885,1.842666,31.356644
10,BTC-USD,1d,NHITS,365,3650,2478,620,2347.910011,1776.668512,2.036757,39.897192
4,BTC-USD,1h,Naive,168,730,14015,3504,458.912642,305.937922,0.321119,0.000000
7,BTC-USD,1h,iTransformer,168,730,14015,3504,460.659781,308.214226,0.323334,28.977906
5,BTC-USD,1h,DLinear,168,730,14015,3504,465.005015,308.435637,0.323793,25.349479
6,BTC-USD,1h,NHITS,168,730,14015,3504,477.196190,329.777674,0.345219,44.325216
0,BTC-USD,1m,Naive,240,7,8063,2016,58.905498,41.079013,0.058332,0.000000
1,BTC-USD,1m,DLinear,240,7,8063,2016,60.393214,42.311425,0.060080,31.351882


Сохранено: registry.csv (строк: 24)


In [ ]:
#@title Итоги: лучшие модели по каждой частоте и тикеру
if 'registry' in globals() and len(registry)>0:
    best = registry.sort_values(["ticker","step","rmse_test"]).groupby(["ticker","step"], as_index=False).first()
    display(best.sort_values(["ticker","step"]))
